In [7]:
import pandas as pd

df = pd.read_csv("delivery_order.csv")

df.head()

,order_id,store_id,city,order_placed_time,order_delivered_time,delivery_minutes,distance_km,promised_sla_minutes,breached,day_of_week,hour_of_day
0,ORD-000001,STR-001,Jalandhar,2026-06-01 08:26:58,2026-06-01 08:37:11.200,10.22,0.81,10,True,Monday,8
1,ORD-000002,STR-001,Jalandhar,2026-06-01 15:51:56,2026-06-01 16:00:47.000,8.85,0.26,10,False,Monday,15
2,ORD-000003,STR-001,Jalandhar,2026-06-01 11:38:17,2026-06-01 11:49:51.200,11.57,3.77,10,True,Monday,11
3,ORD-000004,STR-001,Jalandhar,2026-06-01 10:19:27,2026-06-01 10:30:36.000,11.15,0.90,10,True,Monday,10
4,ORD-000005,STR-001,Jalandhar,2026-06-01 17:46:34,2026-06-01 17:56:09.400,9.59,3.08,10,False,Monday,17


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49944 entries, 0 to 49943
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   order_id              49944 non-null  object 
 1   store_id              49944 non-null  object 
 2   city                  49944 non-null  object 
 3   order_placed_time     49944 non-null  object 
 4   order_delivered_time  49944 non-null  object 
 5   delivery_minutes      49944 non-null  float64
 6   distance_km           49944 non-null  float64
 7   promised_sla_minutes  49944 non-null  int64  
 8   breached              49944 non-null  bool   
 9   day_of_week           49944 non-null  object 
 10  hour_of_day           49944 non-null  int64  
dtypes: bool(1), float64(2), int64(2), object(6)
memory usage: 3.9+ MB


In [10]:
df.shape

(49944, 11)

In [12]:
df['breached'].mean()

np.float64(0.4141838859522665)

In [13]:
import sqlite3

conn = sqlite3.connect("sla_data.db")

df.to_sql("orders", conn, if_exists="replace", index=False)

print("Database created successfully!")

Database created successfully!


In [14]:
cursor = conn.cursor()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

print(cursor.fetchall())

[('orders',)]


In [15]:
query = """
SELECT *
FROM orders
LIMIT 5;
"""

result = pd.read_sql_query(query, conn)

result

,order_id,store_id,city,order_placed_time,order_delivered_time,delivery_minutes,distance_km,promised_sla_minutes,breached,day_of_week,hour_of_day
0,ORD-000001,STR-001,Jalandhar,2026-06-01 08:26:58,2026-06-01 08:37:11.200,10.22,0.81,10,1,Monday,8
1,ORD-000002,STR-001,Jalandhar,2026-06-01 15:51:56,2026-06-01 16:00:47.000,8.85,0.26,10,0,Monday,15
2,ORD-000003,STR-001,Jalandhar,2026-06-01 11:38:17,2026-06-01 11:49:51.200,11.57,3.77,10,1,Monday,11
3,ORD-000004,STR-001,Jalandhar,2026-06-01 10:19:27,2026-06-01 10:30:36.000,11.15,0.90,10,1,Monday,10
4,ORD-000005,STR-001,Jalandhar,2026-06-01 17:46:34,2026-06-01 17:56:09.400,9.59,3.08,10,0,Monday,17


In [16]:
df.head()

,order_id,store_id,city,order_placed_time,order_delivered_time,delivery_minutes,distance_km,promised_sla_minutes,breached,day_of_week,hour_of_day
0,ORD-000001,STR-001,Jalandhar,2026-06-01 08:26:58,2026-06-01 08:37:11.200,10.22,0.81,10,True,Monday,8
1,ORD-000002,STR-001,Jalandhar,2026-06-01 15:51:56,2026-06-01 16:00:47.000,8.85,0.26,10,False,Monday,15
2,ORD-000003,STR-001,Jalandhar,2026-06-01 11:38:17,2026-06-01 11:49:51.200,11.57,3.77,10,True,Monday,11
3,ORD-000004,STR-001,Jalandhar,2026-06-01 10:19:27,2026-06-01 10:30:36.000,11.15,0.90,10,True,Monday,10
4,ORD-000005,STR-001,Jalandhar,2026-06-01 17:46:34,2026-06-01 17:56:09.400,9.59,3.08,10,False,Monday,17


In [18]:
query = """
SELECT 
    store_id,
    COUNT(*) AS total_orders,
    ROUND(
        AVG(CASE WHEN breached = 1 THEN 1.0 ELSE 0 END) * 100,
        1
    ) AS breach_rate_pct
FROM orders
GROUP BY store_id
ORDER BY breach_rate_pct DESC;
"""

result = pd.read_sql_query(query, conn)
result

,store_id,total_orders,breach_rate_pct
0,STR-002,4320,96.2
1,STR-003,4178,90.6
2,STR-001,4162,73.3
3,STR-006,4022,49.3
4,STR-007,4162,46.3
5,STR-004,4125,42.3
6,STR-008,4335,32.9
7,STR-005,4034,26.2
8,STR-012,4211,12.6
9,STR-011,4108,9.5


In [19]:
query2 = """

SELECT 
    hour_of_day,
    ROUND(
        AVG(CASE WHEN breached = 1 THEN 1.0 ELSE 0 END) * 100,
        1
    ) AS breach_rate_pct,
    COUNT(*) AS orders
FROM orders
GROUP BY hour_of_day
ORDER BY breach_rate_pct DESC;
"""

result2 = pd.read_sql_query(query2, conn)
result2

,hour_of_day,breach_rate_pct,orders
0,22,67.6,2744
1,13,66.3,2726
2,21,66.1,4028
3,14,66.1,2765
4,20,65.6,5427
5,10,28.5,2772
6,12,28.1,4044
7,9,28.1,1365
8,19,28.0,5323
9,16,27.9,2727


In [20]:

df.columns


Index(['order_id', 'store_id', 'city', 'order_placed_time',
       'order_delivered_time', 'delivery_minutes', 'distance_km',
       'promised_sla_minutes', 'breached', 'day_of_week', 'hour_of_day'],
      dtype='object')

In [22]:
df[['hour_of_day']].head(10)

,hour_of_day
0,8
1,15
2,11
3,10
4,17
5,17
6,20
7,15
8,13
9,19


In [23]:
query3 = """
SELECT 
    day_of_week,
    ROUND(
        AVG(CASE WHEN breached = 1 THEN 1.0 ELSE 0 END) * 100,
        1
    ) AS breach_rate_pct
FROM orders
GROUP BY day_of_week
ORDER BY breach_rate_pct DESC;
"""

result3 = pd.read_sql_query(query3, conn)
result3

,day_of_week,breach_rate_pct
0,Tuesday,42.8
1,Friday,42.7
2,Thursday,41.4
3,Monday,41.1
4,Sunday,41.0
5,Wednesday,40.7
6,Saturday,40.6


In [24]:
query4 = """
WITH daily_store AS (
    SELECT
        store_id,
        DATE(order_placed_time) AS order_date,
        AVG(CASE WHEN breached = 1 THEN 1.0 ELSE 0 END) AS daily_breach_rate
    FROM orders
    GROUP BY store_id, DATE(order_placed_time)
)

SELECT
    store_id,
    order_date,
    ROUND(
        AVG(daily_breach_rate) OVER (
            PARTITION BY store_id
            ORDER BY order_date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) * 100,
        1
    ) AS rolling_7day_breach_rate_pct
FROM daily_store
ORDER BY store_id, order_date;
"""

result4 = pd.read_sql_query(query4, conn)
result4.head(20)

,store_id,order_date,rolling_7day_breach_rate_pct
0,STR-001,2026-06-01,77.1
1,STR-001,2026-06-02,75.6
2,STR-001,2026-06-03,73.7
3,STR-001,2026-06-04,72.7
4,STR-001,2026-06-05,71.7
5,STR-001,2026-06-06,71.3
6,STR-001,2026-06-07,71.5
7,STR-001,2026-06-08,71.5
8,STR-001,2026-06-09,70.7
9,STR-001,2026-06-10,72.1


In [25]:
query5 = """
SELECT 
    store_id,
    ROUND(AVG(distance_km), 2) AS avg_distance,
    ROUND(AVG(delivery_minutes), 2) AS avg_delivery_min,
    COUNT(*) AS orders
FROM orders
GROUP BY store_id
ORDER BY avg_delivery_min DESC;
"""

result5 = pd.read_sql(query5, conn)
result5


,store_id,avg_distance,avg_delivery_min,orders
0,STR-002,1.36,13.02,4320
1,STR-003,1.41,12.42,4178
2,STR-001,1.39,11.37,4162
3,STR-006,1.41,10.20,4022
4,STR-007,1.40,10.01,4162
5,STR-004,1.40,9.82,4125
6,STR-008,1.45,9.18,4335
7,STR-005,1.41,8.76,4034
8,STR-012,1.41,7.54,4211
9,STR-011,1.39,7.25,4108


In [26]:
query6 = """
SELECT 
    store_id,
    hour_of_day,
    ROUND(
        AVG(CASE WHEN breached = 1 THEN 1.0 ELSE 0 END) * 100,
        1
    ) AS breach_rate_pct
FROM orders
WHERE store_id IN ('STR-002', 'STR-003', 'STR-001')
GROUP BY store_id, hour_of_day
ORDER BY store_id, hour_of_day;
"""

result6 = pd.read_sql(query6, conn)
result6

,store_id,hour_of_day,breach_rate_pct
0,STR-001,8,60.2
1,STR-001,9,55.2
2,STR-001,10,59.2
3,STR-001,11,55.6
4,STR-001,12,57.8
5,STR-001,13,98.3
6,STR-001,14,97.9
7,STR-001,15,60.6
8,STR-001,16,61.9
9,STR-001,17,60.5
